In [ ]:
!date

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from concurrent.futures import ProcessPoolExecutor, as_completed
from tqdm import tqdm

from gprofiler import GProfiler
import gseapy as gp

In [ ]:
projdir = '/u/project/cluo/terencew/igvf/2023_YR2/snmCT/rna'
donors = list(np.loadtxt(f'{projdir}/txt/donors.txt', dtype=str))
tmp_order = ['Start', 'Sendai', 'Fail1', 'Delayed', 'Inter1', 'Inter2', 'Fail2', 'IPS']
clusts = [f'{x}_{y}' for x,y in zip(np.repeat(tmp_order, len(donors)), np.tile(donors, len(tmp_order)))]

In [ ]:
main_degs = pd.read_csv(f'{projdir}/csv/degs/ips_subtypes/ips_main.csv', sep='\t')
alt_degs = pd.read_csv(f'{projdir}/csv/degs/ips_subtypes/ips_alt.csv', sep='\t')
main_degs.shape, alt_degs.shape

In [ ]:
main_degs

In [ ]:
tmp = main_degs.sort_values(by=['logfoldchanges'], ascending=False)

In [ ]:
tmp[tmp['pvals_adj'] < 1e-2]['names'].values[:10]

In [ ]:
thresh = 1e-3
main_filt = main_degs[main_degs['pvals_adj'] < thresh]
main_filt.shape

In [ ]:
thresh = 1e-3
main_filt = main_degs[main_degs['pvals_adj'] < thresh]
main_filt.shape

In [ ]:
thresh = 1e-3
alt_filt = alt_degs[alt_degs['pvals_adj'] < thresh]
alt_filt.shape

In [ ]:
tmp1 = main_filt.sort_values(by=['logfoldchanges', 'pvals_adj'], ascending=False)['names']

In [ ]:
tmp2 = alt_filt.sort_values(by=['logfoldchanges', 'pvals_adj'], ascending=False)['names']

In [ ]:
tmp2.isin(tmp1).sum()

In [ ]:
gp = GProfiler(return_dataframe=True)
genes = list(main_filt['names'].values)
tmp_results = gp.profile(
    organism='hsapiens',
    query=genes,
    sources=['GO:BP', 'GO:MF', 'GO:CC'],
    no_evidences=False
)

tmp_results.shape

In [ ]:
tmp_results.head()

In [ ]:
def plot_go_dot(df, top_n=15, title="GO Enrichment"):
    df = df.copy()

    df["p_value"] = df["p_value"].replace(0, 1e-300)
    df["neglog10_p"] = -np.log10(df["p_value"])

    # gene ratio = intersection / query size
    df["gene_ratio"] = df["intersection_size"] / df["query_size"]

    df_top = df.sort_values("p_value").head(top_n)
    df_top = df_top.iloc[::-1]

    plt.figure(figsize=(6, 5))

    plt.scatter(
        df_top["gene_ratio"],
        df_top["name"],
        s=df_top["intersection_size"] * 5  # size encodes overlap
    )

    plt.xlabel("Gene ratio")
    plt.title(title)

    plt.tight_layout()
    plt.show()

In [ ]:
plot_go_dot(tmp_results)

In [ ]:
outdir = '/u/project/cluo/terencew/igvf/2023_YR2/final_figures/csv/figure5'
tmp_results.to_csv(f'{outdir}/ips_subtypes_go.csv', sep='\t')

In [ ]:
!date